In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import subprocess
subprocess.run([
    'pip', 'install', 'mlflow', 'dagshub', 'scikit-learn', 'pandas',
    'matplotlib', 'seaborn', 'imbalanced-learn', '--quiet'
])

import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET = 'isFraud'
ID_COL = 'TransactionID'
TIME_COL = 'TransactionDT'

In [ ]:
from kaggle_secrets import UserSecretsClient
import dagshub
import mlflow
import mlflow.sklearn

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret('DAGSHUB_TOKEN')

dagshub.auth.add_app_token(token=dagshub_token)
dagshub.init(repo_owner='ngval22', repo_name='fraud-detection-classification', mlflow=True)
print('Tracking URI:', mlflow.get_tracking_uri())

EXPERIMENT_NAME = 'RandomForest_Training'
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name='RF_Parent') as parent_run:
    PARENT_RUN_ID = parent_run.info.run_id
    mlflow.log_param('model_family', 'RandomForest')
    mlflow.log_param('random_state', RANDOM_STATE)
    mlflow.log_param('experiment', EXPERIMENT_NAME)

print('Parent run ID:', PARENT_RUN_ID)

In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train_merged = train_transaction.merge(train_identity, on=ID_COL, how='left')
test_merged  = test_transaction.merge(test_identity,  on=ID_COL, how='left')

y_train     = train_merged[TARGET].copy()
X_raw_train = train_merged.drop(columns=[TARGET])
X_raw_test  = test_merged.copy()
test_ids    = test_transaction[ID_COL].copy()

print('train merged:', train_merged.shape)
print('test merged: ', test_merged.shape)
print(f'Fraud rate: {y_train.mean():.4f}')

# Cleaning

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler


class CleaningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, missing_thresh=0.50, id_col=ID_COL,
                 time_col=TIME_COL, target=TARGET):
        self.missing_thresh = missing_thresh
        self.id_col = id_col
        self.time_col = time_col
        self.target = target

    def fit(self, X, y=None):
        missing_pct = X.isnull().mean()
        self.cols_to_drop_ = [
            c for c in missing_pct[missing_pct > self.missing_thresh].index
            if c not in [self.target, self.id_col]
        ]
        self.common_cols_ = [
            c for c in X.columns
            if c not in self.cols_to_drop_
            and c not in [self.target, self.id_col]
        ]
        return self

    def transform(self, X):
        out = X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns],
                     errors='ignore')
        out = out.drop(columns=[self.id_col], errors='ignore')
        out = out.drop(columns=[self.target], errors='ignore')
        return out[[c for c in self.common_cols_ if c in out.columns]]


cleaning_probe = CleaningTransformer().fit(X_raw_train, y_train)
print('Dropped columns:', len(cleaning_probe.cols_to_drop_))
print('Kept columns:', len(cleaning_probe.common_cols_))

# Feature Engineering

In [ ]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    IMPORTANT_NUM = ['D1','D2','D3','D4','D5','D6',
                     'C1','C2','C3','C4','C5',
                     'M1','M2','M3','M4','M5']

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include='object').columns.tolist()
        self.ord_enc_ = OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1
        )
        if self.cat_cols_:
            self.ord_enc_.fit(X[self.cat_cols_].fillna('__MISSING__'))
        self.missing_flag_cols_ = [c for c in self.IMPORTANT_NUM if c in X.columns]
        return self

    def transform(self, X):
        out = X.copy()

        if 'TransactionAmt' in out.columns:
            out['TransactionAmt_log'] = np.log1p(out['TransactionAmt'])
            out['TransactionAmt_cents'] = (out['TransactionAmt'] % 1).round(2)
            out['Amt_is_round'] = (out['TransactionAmt'] % 1 == 0).astype(int)

        if TIME_COL in out.columns:
            out['hour'] = (out[TIME_COL] // 3600) % 24
            out['day_of_week'] = (out[TIME_COL] // (3600 * 24)) % 7
            out['is_weekend'] = out['day_of_week'].isin([5, 6]).astype(int)
            out = out.drop(columns=[TIME_COL])

        for col in self.missing_flag_cols_:
            out[f'{col}_missing'] = out[col].isnull().astype(int)

        if self.cat_cols_:
            out[self.cat_cols_] = self.ord_enc_.transform(
                out[self.cat_cols_].fillna('__MISSING__')
            )

        return out


fe_probe = FeatureEngineeringTransformer().fit(cleaning_probe.transform(X_raw_train.head(5000)), y_train.head(5000))
print('Categorical columns encoded:', len(fe_probe.cat_cols_))
print('Missing indicator columns:', len(fe_probe.missing_flag_cols_))

# Feature Selection

In [ ]:
class RFFeatureSelectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, corr_thresh=0.98, var_thresh=0.0,
                 n_features=80, sample_size=80000,
                 random_state=RANDOM_STATE):
        self.corr_thresh = corr_thresh
        self.var_thresh = var_thresh
        self.n_features = n_features
        self.sample_size = sample_size
        self.random_state = random_state

    def fit(self, X, y=None):
        self.input_columns_ = X.columns.tolist()

        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X)
        X_imp_df = pd.DataFrame(X_imp, columns=self.input_columns_)

        corr = X_imp_df.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.cols_drop_corr_ = [c for c in upper.columns if any(upper[c] > self.corr_thresh)]
        surviving = [c for c in self.input_columns_ if c not in self.cols_drop_corr_]

        vt = VarianceThreshold(threshold=self.var_thresh)
        vt.fit(X_imp_df[surviving])
        self.cols_keep_vt_ = [f for f, keep in zip(surviving, vt.get_support()) if keep]

        y_series = pd.Series(y).reset_index(drop=True)
        if len(y_series) > self.sample_size:
            per_class_n = min(y_series.value_counts().min(), self.sample_size // 2)
            sample_idx = y_series.groupby(y_series).sample(
                n=per_class_n,
                random_state=self.random_state
            ).index
            X_sample = X_imp_df.iloc[sample_idx][self.cols_keep_vt_]
            y_sample = y_series.iloc[sample_idx]
        else:
            X_sample = X_imp_df[self.cols_keep_vt_]
            y_sample = y_series

        selector_model = RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=5,
            class_weight='balanced_subsample',
            random_state=self.random_state,
            n_jobs=-1,
        )
        selector_model.fit(X_sample, y_sample)

        importances = pd.Series(selector_model.feature_importances_, index=self.cols_keep_vt_)
        self.feature_importances_ = importances.sort_values(ascending=False)
        self.selected_features_ = self.feature_importances_.head(self.n_features).index.tolist()
        return self

    def transform(self, X):
        return X[[c for c in self.selected_features_ if c in X.columns]]

# Full Pipeline

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scorer = 'roc_auc'


def build_rf_pipeline(n_estimators, max_depth, min_samples_leaf,
                      max_features='sqrt', class_weight=None,
                      use_undersampling=False):
    preprocessing = [
        ('cleaning', CleaningTransformer()),
        ('engineering', FeatureEngineeringTransformer()),
        ('selection', RFFeatureSelectionTransformer(random_state=RANDOM_STATE)),
        ('imputer', SimpleImputer(strategy='median')),
    ]

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    if use_undersampling:
        return ImbPipeline(preprocessing + [
            ('under', RandomUnderSampler(random_state=RANDOM_STATE)),
            ('rf', rf),
        ])

    return Pipeline(preprocessing + [('rf', rf)])

In [ ]:
with mlflow.start_run(run_id=PARENT_RUN_ID):
    with mlflow.start_run(run_name='RandomForest_Cleaning', nested=True):
        mlflow.log_params({
            'stage': 'cleaning',
            'missing_threshold': cleaning_probe.missing_thresh,
            'dropped_missing_columns': len(cleaning_probe.cols_to_drop_),
            'kept_columns_after_cleaning': len(cleaning_probe.common_cols_),
            'dropped_id_column': ID_COL,
        })

    with mlflow.start_run(run_name='RandomForest_Feature_Engineering', nested=True):
        mlflow.log_params({
            'stage': 'feature_engineering',
            'amount_features': 'TransactionAmt_log,TransactionAmt_cents,Amt_is_round',
            'time_features': 'hour,day_of_week,is_weekend',
            'categorical_encoding': 'OrdinalEncoder(handle_unknown=use_encoded_value)',
            'categorical_columns': len(fe_probe.cat_cols_),
            'missing_indicator_columns': len(fe_probe.missing_flag_cols_),
            'scaling': 'not_used_for_random_forest',
        })

    with mlflow.start_run(run_name='RandomForest_Feature_Selection', nested=True):
        mlflow.log_params({
            'stage': 'feature_selection',
            'correlation_threshold': 0.98,
            'variance_threshold': 0.0,
            'selection_method': 'RandomForest_feature_importance',
            'selected_features': 80,
            'selector_sample_size': 80000,
        })

print('RandomForest preprocessing stage runs logged.')

# Training

In [ ]:
def evaluate_time_holdout(pipe, holdout_frac=0.20):
    from sklearn.base import clone
    from sklearn.metrics import roc_auc_score

    cutoff = X_raw_train[TIME_COL].quantile(1 - holdout_frac)
    train_mask = X_raw_train[TIME_COL] <= cutoff
    val_mask = X_raw_train[TIME_COL] > cutoff

    time_pipe = clone(pipe)
    time_pipe.fit(X_raw_train.loc[train_mask], y_train.loc[train_mask])
    val_pred = time_pipe.predict_proba(X_raw_train.loc[val_mask])[:, 1]
    return roc_auc_score(y_train.loc[val_mask], val_pred)

def run_rf_experiment(run_name, n_estimators, max_depth, min_samples_leaf,
                      max_features='sqrt', class_weight=None,
                      use_undersampling=False):
    with mlflow.start_run(run_id=PARENT_RUN_ID):
        with mlflow.start_run(run_name=run_name, nested=True) as run:
            pipe = build_rf_pipeline(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                class_weight=class_weight,
                use_undersampling=use_undersampling,
            )

            cv_results = cross_validate(
                pipe, X_raw_train, y_train,
                cv=CV, scoring=scorer,
                return_train_score=True,
                n_jobs=1,
                error_score='raise',
            )

            mean_val_auc = cv_results['test_score'].mean()
            std_val_auc = cv_results['test_score'].std()
            mean_train_auc = cv_results['train_score'].mean()
            overfit_gap = mean_train_auc - mean_val_auc
            time_holdout_auc = evaluate_time_holdout(pipe)

            mlflow.log_params({
                'n_estimators': n_estimators,
                'max_depth': max_depth,
                'min_samples_leaf': min_samples_leaf,
                'max_features': max_features,
                'class_weight': str(class_weight),
                'use_undersampling': use_undersampling,
                'cv_folds': 5,
            })
            mlflow.log_metrics({
                'cv_val_roc_auc_mean': round(mean_val_auc, 5),
                'cv_val_roc_auc_std': round(std_val_auc, 5),
                'cv_train_roc_auc_mean': round(mean_train_auc, 5),
                'overfit_gap': round(overfit_gap, 5),
                'time_holdout_roc_auc': round(time_holdout_auc, 5),
            })
            for i, (tr_s, val_s) in enumerate(zip(
                cv_results['train_score'], cv_results['test_score']
            )):
                mlflow.log_metric(f'fold_{i+1}_train_auc', round(tr_s, 5))
                mlflow.log_metric(f'fold_{i+1}_val_auc', round(val_s, 5))

            print(f'{run_name}: val_AUC={mean_val_auc:.5f} ± {std_val_auc:.5f} | '
                  f'train_AUC={mean_train_auc:.5f} | gap={overfit_gap:.5f} | '
                  f'time_holdout_AUC={time_holdout_auc:.5f}')

            pipe.fit(X_raw_train, y_train)
            mlflow.sklearn.log_model(pipe, name='model')

            return mean_val_auc, std_val_auc, pipe, run.info.run_id

In [ ]:
# Experiment 1: baseline RandomForest
auc1, std1, pipe1, rid1 = run_rf_experiment(
    run_name='RF_01_Baseline',
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight=None,
)

In [ ]:
# Experiment 2: balanced class weights
auc2, std2, pipe2, rid2 = run_rf_experiment(
    run_name='RF_02_Balanced_Weights',
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight='balanced_subsample',
)

In [ ]:
# Experiment 3: shallow model, underfitting check
auc3, std3, pipe3, rid3 = run_rf_experiment(
    run_name='RF_03_Shallow_Underfit',
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=20,
    class_weight='balanced_subsample',
)

In [ ]:
# Experiment 4: deeper model, overfitting check
auc4, std4, pipe4, rid4 = run_rf_experiment(
    run_name='RF_04_Deeper_Overfit_Check',
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced_subsample',
)

In [ ]:
# Experiment 5: undersampling comparison
auc5, std5, pipe5, rid5 = run_rf_experiment(
    run_name='RF_05_UnderSampling',
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight=None,
    use_undersampling=True,
)

In [ ]:
# Experiment 6: regularized RandomForest
auc6, std6, pipe6, rid6 = run_rf_experiment(
    run_name='RF_06_Regularized',
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced_subsample',
)

# Results

In [ ]:
results = [
    {'run': 'RF_01_Baseline',             'val_auc': auc1, 'std': std1, 'run_id': rid1},
    {'run': 'RF_02_Balanced_Weights',     'val_auc': auc2, 'std': std2, 'run_id': rid2},
    {'run': 'RF_03_Shallow_Underfit',     'val_auc': auc3, 'std': std3, 'run_id': rid3},
    {'run': 'RF_04_Deeper_Overfit_Check', 'val_auc': auc4, 'std': std4, 'run_id': rid4},
    {'run': 'RF_05_UnderSampling',        'val_auc': auc5, 'std': std5, 'run_id': rid5},
    {'run': 'RF_06_Regularized',          'val_auc': auc6, 'std': std6, 'run_id': rid6},
]

results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
print(results_df[['run', 'val_auc', 'std']].to_string(index=False))

best_row = results_df.iloc[0]
BEST_PIPE_NAME = best_row['run']
BEST_RUN_ID = best_row['run_id']
print(f'Best RandomForest model: {BEST_PIPE_NAME} (val AUC = {best_row["val_auc"]:.5f})')

In [ ]:
pipeline_map = {
    'RF_01_Baseline': pipe1,
    'RF_02_Balanced_Weights': pipe2,
    'RF_03_Shallow_Underfit': pipe3,
    'RF_04_Deeper_Overfit_Check': pipe4,
    'RF_05_UnderSampling': pipe5,
    'RF_06_Regularized': pipe6,
}

best_pipeline = pipeline_map[BEST_PIPE_NAME]

with mlflow.start_run(run_id=PARENT_RUN_ID):
    mlflow.log_metric('best_val_roc_auc', round(best_row['val_auc'], 5))
    mlflow.log_param('best_child_run', BEST_PIPE_NAME)
    mlflow.log_param('best_run_id', BEST_RUN_ID)

print('Best RandomForest pipeline selected:', BEST_PIPE_NAME)
print(best_pipeline)

In [ ]:
MODEL_REGISTRY_NAME = 'RF_BestPipeline'

with mlflow.start_run(run_id=PARENT_RUN_ID):
    with mlflow.start_run(run_name='RandomForest_Best_Model_Save', nested=True):
        mlflow.log_param('saved_pipeline', BEST_PIPE_NAME)
        mlflow.log_metric('best_val_roc_auc', round(best_row['val_auc'], 5))

        input_example = X_raw_train.head(5)
        model_info = mlflow.sklearn.log_model(
            sk_model=best_pipeline,
            name='best_rf_pipeline',
            input_example=input_example,
        )
        print(f'Best RandomForest pipeline saved as: {MODEL_REGISTRY_NAME}')
        print(f'Model URI: {model_info.model_uri}')